# Using the new sampler classes

Just a simple notebook here where we use the implemented samplers.

## Linear regression

One of the easiest models we can look at is a simple linear regression with a Gaussian prior on the coefficients.
\begin{align}
\pi(\mathbf{y} \vert \boldsymbol{\beta},X) = \mathcal{N}(X\boldsymbol{\beta},\sigma^2I),  \ \ \ \ \pi(\boldsymbol{\beta})=\mathcal{N}(0,\alpha^2I).
\end{align}

All the sampler need is the gradient of the log-posterior 
\begin{align}
\nabla_{\boldsymbol{\beta}} \ U(\boldsymbol{\beta}) &= \nabla_{\boldsymbol{\beta}} \ \left[\log \mathcal{N}(X\boldsymbol{\beta},\sigma^2I) + \log \mathcal{N}(0,\alpha^2I)\right] \\
&= \nabla_{\boldsymbol{\beta}} \ \left[ C(\sigma^2) -\frac{1}{2}\left(\frac{1}{\sigma^2}(\mathbf{y}-X\boldsymbol{\beta})^T(\mathbf{y}-X\boldsymbol{\beta})\right) - \frac{1}{2}\left(\frac{\boldsymbol{\beta}^T\boldsymbol{\beta}}{\alpha^2}\right) \right],
\end{align}
where $C(\sigma^2)$ is some constant that depend on \sigma^2, which we for now take as fixed.

In [ ]:
import numpy as np
# Okay, now onto some actual code
# I need to sample some true coefficients, a design matrix and some actual data
n = 1000
d = 100
alpha = 2.0
sigma = 2.0
X = np.random.randn(n, d)
X = (X - X.mean(0)) / X.std(0)
beta = np.random.normal(0,alpha,size=d)
epsilon = np.random.normal(0,sigma,size=n)
y = X @ beta + epsilon


In [ ]:
beta

In [ ]:
import jax.numpy as jnp
from jax import grad, jit
from functools import partial

def target(beta,y,X,sigma,alpha):
    yhat = jnp.matmul(X, beta)
    res = y-yhat
    restres = jnp.matmul(jnp.transpose(res),res)
    btb = jnp.matmul(jnp.transpose(beta),beta)
    u = (restres / sigma**2) + (btb / alpha**2)
    return 0.5*u

u = partial(target,y = y, X = X, sigma = sigma, alpha = alpha)
d_target = jit(grad(u,argnums=(0)))


In [ ]:
# Now setting up sampler
from sazz.samplers.AutomaticZigZagSampler import AutomaticZigZagSampler

sampler = AutomaticZigZagSampler(N=20000,D=d,grad_target=d_target,gamma=0.000)
sampler.sample()

In [ ]:
import matplotlib.pyplot as plt
Position = sampler.Position
i1 = 0
i2 = 1
plt.plot(Position[:,i1], Position[:,i2])
plt.axvline(beta[i1],color="black",alpha=0.3)
plt.axhline(beta[i2],color="black",alpha=0.3)
plt.xlabel("beta_"+str(i1))
plt.ylabel("beta_"+str(i2))


In [ ]:
points = sampler.getSamples(N_samples = 40000)

In [ ]:
plt.scatter(points[30000:, i1], points[30000:, i2], alpha=0.5, label="Samples",s=0.5)
plt.axvline(beta[i1],color="black",alpha=0.3)
plt.axhline(beta[i2],color="black",alpha=0.3)

In [ ]:
plt.plot(sampler.Time)

In [ ]:
sampler.Position.max()

In [ ]:
sampler.t_max

## Sticky Automatic ZigZag

In [ ]:
import numpy as np
# Okay, now onto some actual code
# I need to sample some true coefficients, a design matrix and some actual data
n = 1000
d = 2000
d0 = 5 # Active components
alpha = 2.0
sigma = 2.0
X = np.random.randn(n, d)
X = (X - X.mean(0)) / X.std(0)
beta_a = np.random.normal(0,alpha,size=d0)
beta = np.concat((beta_a,np.zeros(d-d0)))
epsilon = np.random.normal(0,sigma,size=n)
y = X @ beta + epsilon

# And we need the kappa parameters for freezing times
weights = np.full((d),d0/(d),float)
dens_at_zero = 1 / (np.sqrt(2 * np.pi) * alpha)
kappa = weights/(1-weights)*dens_at_zero

In [ ]:
np.sum(X**2, axis=0)/sigma**2 + 1/alpha**2

In [ ]:
import jax.numpy as jnp
from jax import grad, jit
from functools import partial

def target(beta,y,X,sigma,alpha):
    yhat = jnp.matmul(X, beta)
    res = y-yhat
    restres = jnp.matmul(jnp.transpose(res),res)
    btb = jnp.matmul(jnp.transpose(beta),beta)
    u = (restres / sigma**2) + (btb / alpha**2)
    return 0.5*u

u = partial(target,y = y, X = X, sigma = sigma, alpha = alpha)
d_target = jit(grad(u,argnums=(0)))


In [ ]:
# Now setting up sampler
from sazz.samplers.StickyAutomaticZigZagSampler import StickyAutomaticZigZagSampler

sampler = StickyAutomaticZigZagSampler(N=10000,D=d,grad_target=d_target, kappa=kappa, gamma=0.000)
sampler.Velocity[0,:] = 0.1*sampler.Velocity[0,:]
sampler.sample()

In [ ]:
import matplotlib.pyplot as plt
Position = sampler.Position
i1 = 1
i2 = 2
plt.plot(Position[:,i1], Position[:,i2])
plt.axvline(beta[i1],color="black",alpha=0.3)
plt.axhline(beta[i2],color="black",alpha=0.3)
plt.xlabel("beta_"+str(i1))
plt.ylabel("beta_"+str(i2))
plt.scatter(Position[0,i1], Position[0,i2],color="red")

In [ ]:
plt.plot(Position[:,i1])

In [ ]:
plt.scatter(points[20000:, i1], points[20000:, i2], alpha=0.5, label="Samples",s=0.5)
plt.axvline(beta[i1],color="black",alpha=0.3)
plt.axhline(beta[i2],color="black",alpha=0.3)

## Logistic regression

In [ ]:
import numpy as np
# Okay, now onto some actual code
# I need to sample some true coefficients, a design matrix and some actual data
n = 2000
d = 500
d0 = 5 # Active components
alpha_draw = 10
alpha = 10

X = np.random.randn(n, d)
X = (X - X.mean(0)) / X.std(0)
beta_a = np.random.normal(0,alpha_draw,size=d0)
beta = np.concat((beta_a,np.zeros(d-d0))).squeeze()
linear_predictor = X @ beta
p = 1.0/(1+np.exp(-linear_predictor))
y = np.random.binomial(1,p,n)

#y = X @ beta + epsilon

# And we need the kappa parameters for freezing times
#weights = np.full((d),d0/(d),float) # Expected sparsity level
weights = np.full(d, 0.1)  # or 0.01
dens_at_zero = 1 / (np.sqrt(2 * np.pi) * alpha)
kappa = weights/(1-weights)*dens_at_zero
#kappa = 1 * np.sqrt(n) * weights / (1 - weights)

In [ ]:
beta_a

In [ ]:
np.random.exponential(1.0 / kappa[1].item())

In [ ]:
import nest_asyncio
nest_asyncio.apply()
import stan
# First let's run the model in Stan to see what we are working with
stancode = """
data {
  int<lower=0> n;   // number of data items
  int<lower=0> p;   // number of predictors
  matrix[n, p] X;   // predictor matrix
  array[n] int<lower=0, upper=1> y;
}
parameters {
  vector[p] beta;       // coefficients for predictors
  vector<lower=0>[p] lambda;
  real<lower=0> tau;
  //real alpha;
}
model {
  lambda ~ cauchy(0,1);
  tau ~ cauchy(0,1/sqrt(p));
  beta ~ normal(0,tau * lambda);
  y ~ bernoulli_logit(X * beta);
  
}
"""
stan_data = {"n": n, "p": d, "X": X, "y": y}
posterior = stan.build(stancode, data=stan_data)

In [ ]:
fit = posterior.sample(num_chains=4, num_samples=1000)

In [ ]:
samples = fit["beta"][:,3000:4000]  # array with shape (8, 4000)

In [ ]:
np.min(Z)

In [ ]:
levels

In [ ]:
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
eps = 0
i1 = 0
i2 = 1
nx = 100
b1 = samples[i1,:]
b2 = samples[i2,:]
xy = np.stack([b1,b2])
kde = gaussian_kde(xy)
# grid bounds with small margin
xmin, xmax = np.min(b1), np.max(b1)
ymin, ymax = np.min(b2), np.max(b2)
xpad = (xmax - xmin) * 0.1 if xmax > xmin else 1.0
ypad = (ymax - ymin) * 0.1 if ymax > ymin else 1.0
xs = np.linspace(xmin - xpad, xmax + xpad, nx)
ys = np.linspace(ymin - ypad, ymax + ypad, nx)
X_grid, Y_grid = np.meshgrid(xs, ys)
grid_coords = np.vstack([X_grid.ravel(), Y_grid.ravel()])

Z = kde(grid_coords).reshape(X_grid.shape)
levels = np.linspace(np.min(Z)+eps, np.max(Z), 7)

# plot
fig, ax = plt.subplots()

ax.contour(X_grid, Y_grid, Z, levels=levels[1:])

plt.show()

In [ ]:
levels[1:]

In [ ]:
beta

In [ ]:
# Now log-likelihood

import jax
import jax.numpy as jnp
from jax import grad, jit
from functools import partial

def target(beta, temperature, y, X, alpha):
    eta = jnp.matmul(X, beta)              # (n,)
    log_lik = jnp.sum(y * jax.nn.log_sigmoid(eta) + (1-y) * jax.nn.log_sigmoid(-eta))
    prior = -(beta @ beta) / (2*alpha**2)
    return -(temperature * log_lik + prior)

u = partial(target,y = y, X = X, alpha = alpha)
d_target = jit(grad(u,argnums=(0)))

In [ ]:
# Now setting up sampler
from sazz.samplers.StickyAutomaticZigZagSampler import StickyAutomaticZigZagSampler
from sazz.samplers.AutomaticZigZagSampler import AutomaticZigZagSampler

sampler = StickyAutomaticZigZagSampler(N=10000,D=d,grad_target=d_target, kappa=kappa, gamma=0.000,t0=00.000000001)
#sampler = AutomaticZigZagSampler(N=10000,D=d,grad_target=d_target, gamma=0.0)
#sampler.Position[0,:] = beta_map + np.abs(np.random.normal(0,0.00000001,size=d))
#sampler.Position[0,:] = beta + np.abs(np.random.normal(0,0.00000001,size=d)) # This works, should also try tempering
#sampler.Velocity[0,:] = -1.0
#sampler.Position[0,:] = np.random.normal(0,0.1,size=d)
sampler.sample() 

In [ ]:
np.where(sampler.active)

In [ ]:
sampler.cause[538]

In [ ]:
import matplotlib.pyplot as plt
Position = sampler.Position
i1 = 4
i2 = 2

In [ ]:
nx = 100
b1 = samples[i1,:]
b2 = samples[i2,:]
xy = np.stack([b1,b2])
kde = gaussian_kde(xy)
# grid bounds with small margin
xmin, xmax = np.min(b1), np.max(b1)
ymin, ymax = np.min(b2), np.max(b2)
xpad = (xmax - xmin) * 0.1 if xmax > xmin else 1.0
ypad = (ymax - ymin) * 0.1 if ymax > ymin else 1.0
xs = np.linspace(xmin - xpad, xmax + xpad, nx)
ys = np.linspace(ymin - ypad, ymax + ypad, nx)
X_grid, Y_grid = np.meshgrid(xs, ys)
grid_coords = np.vstack([X_grid.ravel(), Y_grid.ravel()])

Z = kde(grid_coords).reshape(X_grid.shape)
levels = np.linspace(np.min(Z)+eps, np.max(Z), 7)

In [ ]:
plt.plot(Position[:,i1], Position[:,i2])
plt.axvline(beta[i1],color="black",alpha=0.3)
plt.axhline(beta[i2],color="black",alpha=0.3)
plt.xlabel("beta_"+str(i1))
plt.ylabel("beta_"+str(i2))
plt.scatter(Position[0,i1], Position[0,i2],color="red")
plt.scatter(Position[-1,i1], Position[-1,i2],color="blue")
plt.contour(X_grid, Y_grid, Z, levels=levels[1:])
plt.xlim(np.min(Position[:,i1]),np.max(Position[:,i1]))
plt.ylim(np.min(Position[:,i2]),np.max(Position[:,i2]))

In [ ]:
i1 = 4
plt.plot(sampler.Time,Position[:,i1])
plt.axhline(beta[i1],color="red")
plt.ylabel("beta_"+str(i1))
plt.xlabel("Time")

## Temperature

In [ ]:
import numpy as np
t0 = 20.0

t = 19.0
s= np.clip(t/t0,0.0,1.0)**2


In [ ]:
s